In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving All_Clickstream.csv to All_Clickstream.csv
User uploaded file "All_Clickstream.csv" with length 615443 bytes


In [ ]:
import pandas as pd

df = pd.read_csv('All_Clickstream.csv')
print(df.head())
print(df.columns)
print(df.shape)

  Profile Name    Source Navigation Level Referrer Url Webpage Url  \
0       User 4  Source 0     signupPrompt          NaN         NaN   
1       User 4  Source 0     browseTitles          NaN         NaN   
2       User 4  Source 1     movieDetails          NaN         NaN   
3       User 4  Source 1     browseTitles          NaN         NaN   
4       User 4  Source 1     movieDetails          NaN         NaN   

          Click Utc Ts  
0  2022-10-08 14:04:47  
1  2022-10-08 14:04:46  
2  2022-09-30 14:53:50  
3  2022-09-30 14:53:50  
4  2022-09-30 14:53:49  
Index(['Profile Name', 'Source', 'Navigation Level', 'Referrer Url',
       'Webpage Url', 'Click Utc Ts'],
      dtype='object')
(12104, 6)


In [ ]:
df.isnull().sum()

,0
Profile Name,0
Source,0
Navigation Level,0
Referrer Url,12102
Webpage Url,12099
Click Utc Ts,0


In [ ]:
df['Click Utc Ts'] = pd.to_datetime(df['Click Utc Ts'])

In [ ]:
df = df.sort_values(by='Click Utc Ts')

In [ ]:
display(df.head())

,Profile Name,Source,Navigation Level,Referrer Url,Webpage Url,Click Utc Ts
1746,User 3,Source 0,browseTitles,NaN,NaN,2022-07-28 16:15:39
1745,User 3,Source 0,profilesGate,NaN,NaN,2022-07-28 16:15:39
1744,User 3,Source 0,movieDetails,NaN,NaN,2022-07-28 16:15:57
1743,User 3,Source 0,browseTitles,NaN,NaN,2022-07-28 16:15:57
1742,User 3,Source 0,playback,NaN,NaN,2022-07-28 16:16:31


In [ ]:
df_stream = df[['Profile Name', 'Source', 'Navigation Level', 'Click Utc Ts']].copy()

df_stream.columns = ['user_id', 'source', 'navigation', 'event_time']

df_stream['event_time'] = pd.to_datetime(df_stream['event_time'])

df_stream = df_stream.sort_values(by='event_time')

print(df_stream.head())
print(df_stream.shape)

     user_id    source    navigation          event_time
1746  User 3  Source 0  browseTitles 2022-07-28 16:15:39
1745  User 3  Source 0  profilesGate 2022-07-28 16:15:39
1744  User 3  Source 0  movieDetails 2022-07-28 16:15:57
1743  User 3  Source 0  browseTitles 2022-07-28 16:15:57
1742  User 3  Source 0      playback 2022-07-28 16:16:31
(12104, 4)


In [ ]:
df_stream.isnull().sum()

,0
user_id,0
source,0
navigation,0
event_time,0


In [ ]:
df_stream['event_time'].min(), df_stream['event_time'].max()


(Timestamp('2022-07-28 16:15:39'), Timestamp('2023-06-19 16:07:39'))

In [ ]:
import time
time.sleep(0.5)

In [ ]:
df_stream['time_diff'] = df_stream['event_time'].diff().dt.total_seconds()
df_stream['time_diff'] = df_stream['time_diff'].fillna(0)

print(df_stream[['event_time','time_diff']].head())

              event_time  time_diff
1746 2022-07-28 16:15:39        0.0
1745 2022-07-28 16:15:39        0.0
1744 2022-07-28 16:15:57       18.0
1743 2022-07-28 16:15:57        0.0
1742 2022-07-28 16:16:31       34.0


In [ ]:
compression_factor = 60  # 60x faster replay
df_stream['replay_delay'] = df_stream['time_diff'] / compression_factor

print(df_stream[['time_diff','replay_delay']].head())

      time_diff  replay_delay
1746        0.0      0.000000
1745        0.0      0.000000
1744       18.0      0.300000
1743        0.0      0.000000
1742       34.0      0.566667


In [ ]:
df_stream['time_diff'].describe()

,time_diff
count,1.210400e+04
mean,2.326993e+03
std,1.060361e+05
min,0.000000e+00
25%,0.000000e+00
50%,1.000000e+00
75%,2.100000e+01
max,1.162680e+07


In [ ]:
df_stream['time_diff'].max()

11626800.0

In [ ]:
df_stream['replay_delay'] = df_stream['time_diff'] / compression_factor

# Cap maximum delay to 2 seconds
df_stream['replay_delay'] = df_stream['replay_delay'].clip(upper=2)

In [ ]:
df_stream['replay_delay'].describe()

,replay_delay
count,12104.000000
mean,0.438147
std,0.756014
min,0.000000
25%,0.000000
50%,0.016667
75%,0.350000
max,2.000000


In [ ]:
!pip install azure-eventhub

In [ ]:
import json
import time
from azure.eventhub import EventHubProducerClient, EventData

connection_str = """Endpoint=sb://streaming-platform-ns.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=/w8sqfOoCInXxi+PfsUQwAiND5jP5/iry+AEhAzNsIw="""
eventhub_name = "clcikstreamhub"

producer = EventHubProducerClient.from_connection_string(
    conn_str=connection_str,
    eventhub_name=eventhub_name
)

print("Starting streaming...")

for _, row in df_stream.iterrows():
    event = {
        "user_id": row["user_id"],
        "source": row["source"],
        "navigation": row["navigation"],
        "event_time": row["event_time"].isoformat()
    }

    batch = producer.create_batch()
    batch.add(EventData(json.dumps(event)))
    producer.send_batch(batch)

    time.sleep(row["replay_delay"])  # realistic replay timing

producer.close()

print("Streaming completed.")

Starting streaming...


KeyboardInterrupt: 